# Mistral-Nemo Backend Server
This notebook loads **Mistral-Nemo-Instruct-2407** and exposes it as an HTTP API via **ngrok**.

Run all cells top to bottom. Once the tunnel URL is printed, your Streamlit app will find it automatically using your ngrok API key.

In [5]:
# Install required packages
!pip install -q pyngrok flask transformers accelerate

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model (this takes a few minutes on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
)

print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully!


In [2]:
def generate_text(prompt: str, max_new_tokens: int = 300, temperature: float = 0.7,
                  top_k: int = 50, top_p: float = 0.95) -> str:
    """Run inference with Mistral-Nemo and return the generated text."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=top_k,
            top_p=top_p,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Quick smoke test
test_output = generate_text("Hello, who are you?", max_new_tokens=50)
print("Smoke test output:\n", test_output)

Smoke test output:
 Hello, who are you? And where are you writing from?

I’m an 18-year-old high school student from the Philippines. I’ve been a huge fan of music since I was in grade school, and I’ve been playing guitar for about five years. I’m


In [3]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route("/", methods=["GET"])
def health_check():
    """Simple health check endpoint."""
    return jsonify({"status": "ok", "model": "mistralai/Mistral-Nemo-Instruct-2407"})

@app.route("/generate", methods=["POST"])
def generate():
    """
    Expected request body (JSON):
    {
        "prompt":         "...",
        "max_new_tokens": 300,     # optional, default 300
        "temperature":    0.7,     # optional
        "top_k":          50,      # optional
        "top_p":          0.95     # optional
    }

    Response:
    { "generated_text": "..." }
    """
    data = request.get_json(force=True)

    prompt         = data.get("prompt", "")
    max_new_tokens = int(data.get("max_new_tokens", 300))
    temperature    = float(data.get("temperature", 0.7))
    top_k          = int(data.get("top_k", 50))
    top_p          = float(data.get("top_p", 0.95))

    if not prompt:
        return jsonify({"error": "prompt field is required"}), 400

    try:
        generated = generate_text(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )
        return jsonify({"generated_text": generated})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Start Flask in a background thread so it doesn't block the notebook
flask_thread = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=5000, use_reloader=False))
flask_thread.daemon = True
flask_thread.start()

print("Flask server started on port 5000")

Flask server started on port 5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit


In [ ]:
import subprocess
import time
import requests

NGROK_AUTH_TOKEN = "" #Put your ngrok api key here
NGROK_BIN = "/root/.config/ngrok/ngrok"

# Kill any leftover ngrok processes
subprocess.run(["pkill", "-f", NGROK_BIN], capture_output=True)
time.sleep(2)

# Start ngrok using the full binary path
proc = subprocess.Popen(
    [NGROK_BIN, "http", "5000", "--authtoken", NGROK_AUTH_TOKEN],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"ngrok started (pid {proc.pid}), waiting for it to connect...")

# Poll until port 4040 is ready (up to 30 seconds)
tunnel_url = ""
for i in range(15):
    time.sleep(2)
    try:
        resp = requests.get("http://localhost:4040/api/tunnels", timeout=3)
        tunnels = resp.json().get("tunnels", [])
        if tunnels:
            tunnel_url = next(
                (t["public_url"] for t in tunnels if t["public_url"].startswith("https")),
                tunnels[0]["public_url"]
            )
            break
        print(f"Agent up, waiting for tunnel... ({i+1})")
    except Exception:
        print(f"Waiting for ngrok to start... ({i+1})")

if tunnel_url:
    print("=" * 60)
    print(f"Tunnel URL : {tunnel_url}")
    print("=" * 60)
    print("Your Streamlit app will detect this URL automatically.")
    print("Keep this notebook running while using the Streamlit app.")
else:
    print("ERROR: ngrok failed to create a tunnel after 30 seconds.")
    # Print any output from the process for debugging
    result = subprocess.run(
        [NGROK_BIN, "http", "5000", "--authtoken", NGROK_AUTH_TOKEN, "--log", "stdout"],
        capture_output=True, text=True, timeout=5
    )
    print(result.stdout)
    print(result.stderr)


ngrok started (pid 232), waiting for it to connect...
Tunnel URL : https://pseudoindependently-uninterpreted-fairy.ngrok-free.dev
Your Streamlit app will detect this URL automatically.
Keep this notebook running while using the Streamlit app.


In [13]:
# ── Keep-alive cell ──────────────────────────────────────────────────────────
# Kaggle sessions time out after ~60 minutes of inactivity.
# Run this cell to keep the session alive while the Streamlit app is in use.

import time

print("Keep-alive running. Interrupt the kernel to stop.")
while True:
    time.sleep(60)
    print("Still alive... tunnel is active.")

Keep-alive running. Interrupt the kernel to stop.
Still alive... tunnel is active.
Still alive... tunnel is active.
Still alive... tunnel is active.
Still alive... tunnel is active.
Still alive... tunnel is active.
Still alive... tunnel is active.
Still alive... tunnel is active.
Still alive... tunnel is active.
Still alive... tunnel is active.


127.0.0.1 - - [21/Jul/2026 22:01:18] "POST /generate HTTP/1.1" 200 -


Still alive... tunnel is active.
Still alive... tunnel is active.


KeyboardInterrupt: 